# Calculate Percent of Contig Length Covered for HPRC Contigs (After Alignment with STAR)

## Set up Environment

Load libraries

In [2]:
library(data.table)  
library(pheatmap)

Set working directory as directory with STAR output bedgraph files

In [ ]:
setwd("star_out_path/cohort_results")

## Step 1: Load TRUE contig lengths from STAR index  

In [ ]:
star_index_dir <- "hprc_contigs_star_index"

contig_names <- fread(file.path(star_index_dir, "chrName.txt"),   
                      header = FALSE, col.names = "chrom")  
contig_lengths <- fread(file.path(star_index_dir, "chrLength.txt"),   
                        header = FALSE, col.names = "length")

reference_contigs <- data.table(  
  chrom = contig_names$chrom,  
  length = contig_lengths$length  
)


In [4]:
cat("Reference: Min =", min(reference_contigs$length),   
    "Max =", max(reference_contigs$length),   
    "Total contigs =", nrow(reference_contigs), "\n")

Reference: Min = 806 Max = 152806 Total contigs = 53983 


## Step 2: Set up function to calculate coverage per CONTIG 

### Strand specific analysis

In [5]:
process_sample_by_contig_str <- function(sample_dir, reference_contigs, T_threshold) {
  cat("Processing", sample_dir, "...\n")

  str1_file <- file.path(sample_dir, paste0(sample_dir, "_Signal.Unique.str1.out.bg"))
  str2_file <- file.path(sample_dir, paste0(sample_dir, "_Signal.Unique.str2.out.bg"))

  if (!file.exists(str1_file) || !file.exists(str2_file)) {
    cat("  Warning: files not found\n")
    return(NULL)
  }

  tryCatch({

    # Helper function to process one strand
    process_strand <- function(file, strand_label) {
      bg <- fread(file, col.names = c("chrom", "start", "end", "value"))
      bg[, width := end - start]

      coverage_per_contig <- bg[, .(
        bases_above_T  = sum(width[value >= T_threshold]),
        total_bg_bases = sum(width)
      ), by = chrom]

      result <- merge(reference_contigs[, .(chrom, length)],
                      coverage_per_contig,
                      by = "chrom",
                      all.x = TRUE)

      result[is.na(bases_above_T), bases_above_T := 0]
      result[is.na(total_bg_bases), total_bg_bases := 0]
      result[, pct_covered := (bases_above_T / length) * 100]
      result$sample    <- sample_dir
      result$threshold <- T_threshold
      result$strand    <- strand_label

      return(result)
    }

    # Process each strand separately
    result_str1 <- process_strand(str1_file, "str1")
    result_str2 <- process_strand(str2_file, "str2")

    # Combine both strands into one table
    result <- rbindlist(list(result_str1, result_str2))

    return(result)

  }, error = function(e) {
    cat("  Error:", e$message, "\n")
    return(NULL)
  })
}

## STEP 3: Identify coverage per sample per contig for multiple Ts

In [ ]:
sample_dirs <- list.dirs(path = ".", recursive = FALSE, full.names = FALSE)
head(sample_dirs)

In [ ]:
thresholds <- c(1, 5, 10, 100, 500, 1000, 2000, 5000, 10000, 50000)#

all_results <- lapply(thresholds, function(T_val) {  
  cat("\n=== Threshold:", T_val, "===\n")  
  res <- lapply(sample_dirs, process_sample_by_contig_str,  
                reference_contigs = reference_contigs,  
                T_threshold = T_val)  
  rbindlist(res[!sapply(res, is.null)])  
})

combined <- rbindlist(all_results)  


## Reshape and save simplified format

In [13]:
combined[, threshold_label := paste0("pct_covered_T", threshold, "_", strand)]

coverage_wide <- as.data.frame(dcast(combined,  
                       chrom + length + sample ~ threshold_label,  
                       value.var = "pct_covered"))  

In [ ]:
write.csv(coverage_wide, "~/hprc_contig_pct_cohort.csv",
         row.names=TRUE)

## Investigate RPM range in the cohort - Calculate Basic Stats

In [ ]:
rpm_summary <- lapply(sample_dirs, function(s) {  
  f1 <- file.path(s, paste0(s, "_Signal.Unique.str1.out.bg"))  
  f2 <- file.path(s, paste0(s, "_Signal.Unique.str2.out.bg"))  
    
  if (!file.exists(f1) || !file.exists(f2)) return(NULL)  
    
  bg1 <- fread(f1, col.names = c("chrom", "start", "end", "value"))  
  bg2 <- fread(f2, col.names = c("chrom", "start", "end", "value"))  
    
  data.table(  
    sample     = s,  
    # Strand 1  
    str1_min    = min(bg1$value),  
    str1_q25    = quantile(bg1$value, 0.25),  
    str1_median = median(bg1$value),  
    str1_mean   = mean(bg1$value),  
    str1_q75    = quantile(bg1$value, 0.75),  
    str1_q95    = quantile(bg1$value, 0.95),  
    str1_q99    = quantile(bg1$value, 0.99),  
    str1_max    = max(bg1$value),  
    str1_rows   = nrow(bg1),  
    # Strand 2  
    str2_min    = min(bg2$value),  
    str2_q25    = quantile(bg2$value, 0.25),  
    str2_median = median(bg2$value),  
    str2_mean   = mean(bg2$value),  
    str2_q75    = quantile(bg2$value, 0.75),  
    str2_q95    = quantile(bg2$value, 0.95),  
    str2_q99    = quantile(bg2$value, 0.99),  
    str2_max    = max(bg2$value),  
    str2_rows   = nrow(bg2)  
  )  
})

rpm_summary <- rbindlist(rpm_summary[!sapply(rpm_summary, is.null)])

# Print summary for each strand  
cat("=== Strand 1 (str1) ===\n")  
cat("Min across cohort: ", min(rpm_summary$str1_min), "\n")  
cat("Median of medians: ", median(rpm_summary$str1_median), "\n")  
cat("Mean of means:     ", round(mean(rpm_summary$str1_mean), 2), "\n")  
cat("Max across cohort: ", max(rpm_summary$str1_max), "\n")

cat("\n=== Strand 2 (str2) ===\n")  
cat("Min across cohort: ", min(rpm_summary$str2_min), "\n")  
cat("Median of medians: ", median(rpm_summary$str2_median), "\n")  
cat("Mean of means:     ", round(mean(rpm_summary$str2_mean), 2), "\n")  
cat("Max across cohort: ", max(rpm_summary$str2_max), "\n")

# Compare strands side by side  
cat("\n=== Strand Comparison ===\n")  
#print(rpm_summary[, .(sample, str1_min, str2_min, str1_median, str2_median, str1_max, str2_max)]) 